Q1 What is total revenue for quarter 1996-Q1, and how does it compare to the previous
quarter?


In [0]:
%sql
WITH quarterly_summary AS (
    SELECT 
        quarter_start, 
        SUM(net_revenue) AS total_revenue
    FROM workspace.gold.revenue_metrics
    WHERE quarter_start < '1998-07-01'
    GROUP BY quarter_start
)
SELECT 
    quarter_start,
    total_revenue,
    LAG(total_revenue) OVER (ORDER BY quarter_start) AS prev_quarter_revenue,
    ROUND(
        (total_revenue - LAG(total_revenue) OVER (ORDER BY quarter_start)) 
        / LAG(total_revenue) OVER (ORDER BY quarter_start) * 100, 
    2) AS revenue_qoq_pct
FROM quarterly_summary
WHERE quarter_start IN ('1995-10-01', '1996-01-01')
ORDER BY quarter_start;

A slight decline in revenue of 0.66% is observed, indicating a stable stagnation of figures at around 4.1 billion, without sharp fluctuations.

Q2 How many active customers (at least one order in the last year of data) were there as of
1998-08-02?


In [0]:
%sql
SELECT COUNT(c_custkey) AS active_customers_count
FROM workspace.gold.customer_activity
WHERE is_active_trailing_12m = TRUE;

Q3 If revenue dropped 20% week over week, which alert rule would catch it? Show the query
and demonstrate it against artificially reduced data

In [0]:
%sql
WITH weekly_revenue AS (
    SELECT 
        week_start, 
        SUM(net_revenue) AS current_rev
    FROM workspace.gold.revenue_metrics
    GROUP BY week_start
)
SELECT 
    w1.week_start AS current_week,
    w1.current_rev,
    w2.current_rev AS previous_rev,
    ((w1.current_rev - w2.current_rev) / w2.current_rev) * 100 AS drop_percentage
FROM weekly_revenue w1
JOIN weekly_revenue w2 
    ON w1.week_start = date_add(w2.week_start, 7)
WHERE ((w1.current_rev - w2.current_rev) / w2.current_rev) <= -0.20
ORDER BY w1.week_start;

In [0]:
%sql
--Створюємо тимчасовий набір даних, де для одного з тижнів штучно обрізаємо дохід на 50%
WITH artificially_reduced_data AS (
    SELECT 
        week_start,
        CASE 
            WHEN week_start = '1998-05-04' THEN SUM(net_revenue) * 0.5 -- штучне падіння на 50%
            ELSE SUM(net_revenue)
        END AS net_revenue
    FROM workspace.gold.revenue_metrics
    GROUP BY week_start
),

weekly_revenue AS (
    SELECT 
        week_start, 
        net_revenue AS current_rev
    FROM artificially_reduced_data
)
SELECT 
    w1.week_start AS current_week,
    w1.current_rev,
    w2.current_rev AS previous_rev,
    ((w1.current_rev - w2.current_rev) / w2.current_rev) * 100 AS drop_percentage
FROM weekly_revenue w1
JOIN weekly_revenue w2 
    ON w1.week_start = date_add(w2.week_start, 7)
WHERE ((w1.current_rev - w2.current_rev) / w2.current_rev) <= -0.20
ORDER BY w1.week_start;

The alert triggers if the relative change in revenue between the current and previous week is less than or equal to -20% (drop_percentage <= -20).
When revenue for the week of 1998-05-04 drops by 50%, the query successfully detects this anomalous week and returns a decline of -50.43%, fully satisfying the alert trigger condition.

Q4 Which 3 metrics best represent “company health” at a glance, and why these three?

Net Revenue Trend (adjusted for discounts): Allows for an assessment of the business's overall financial stability and its long-term profitability.

Active Customers (T12M): A key demand indicator that reflects customer engagement and the health of the customer base, independent of seasonal fluctuations.

Contribution of Key Suppliers and Products: Helps identify supply chain risks and critical dependencies on specific products or vendors in a timely manner.